# Teste isolado — ABRACE Energia (Notícias)

Fonte candidata: **ABRACE Energia — Associação Brasileira de Grandes
Consumidores Industriais de Energia**, setor Energia e Gás. Notebook
**descartável** (Fase 1) — sem dispatcher, sem `atualizar_status_fonte`,
sem gravar nada. Só valida:

1. Scraping da listagem (título, resumo, link)
2. Extração de texto completo + descoberta de onde a data aparece

## Confirmado antes de assumir

WordPress padrão confirmado (`robots.txt` simples, sem bloqueios
relevantes na listagem). Tema com grid de posts em `div.post-item` --
título/link em `h5.post-title a`, resumo em `p.from_the_blog_excerpt`.
Paginação `/noticias/page/N/`, 12 posts por página (bate com as 40
páginas indicadas no pedido).

**Sobre a data**: confirmado que não aparece na listagem (nem nos
metadados dos itens `div.post-item`) -- só na página individual, em
`<time class="entry-date published updated" datetime="...">`, com o
atributo `datetime` já em ISO 8601 (`2026-08-07T17:27:23-03:00`), então
não precisa de regex, só fatiar os 10 primeiros caracteres.

**Sobre o título**: a página individual tem **dois** `<h1>` -- o primeiro
é o cabeçalho genérico do template ("Notícias", igual em toda página do
blog), o segundo é o título de verdade (`h1.entry-title`). O extrator
genérico do dispatcher (`extrair_titulo_h1`) pega o *primeiro* `<h1>` da
página -- aqui pegaria "Notícias" errado. Por isso usa o título já obtido
na listagem (`extrair_titulo: None` no `CONFIGS_FONTES`, mesmo padrão de
`agesan_noticias`/`agetransp`), sem tentar reabrir o H1 da página de
detalhe.

**Sobre o texto completo**: `.entry-content` -- já faz parte do
`SELETORES_CONTEUDO` compartilhado do dispatcher (usado por ABAR/Trata
Brasil) -- não precisa de nenhum seletor novo.

Conteúdo é análise/posicionamento institucional (nota de imprensa,
artigo de opinião, cobertura de eventos/leilões) -- sem filtro de
relevância aqui, como pedido; isso é responsabilidade da etapa de NLP.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://abrace.org.br/noticias/"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
ITENS_POR_PAGINA_ESPERADO = 12

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação)

`div.post-item` por post -- título/link em `h5.post-title a`, resumo em
`p.from_the_blog_excerpt` (não usado nos metadados, só pra conferência).
Sem data na listagem (confirmado) -- `published_at` fica `None` aqui,
preenchido só no Teste 2 (abrindo a página individual).

In [0]:
def listar_abrace(max_paginas: int = 4) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = SITE_URL if pagina == 1 else f"{SITE_URL.rstrip('/')}/page/{pagina}/"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página {pagina}; parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        itens_pagina = soup.select("div.post-item")
        if not itens_pagina:
            print(f"  -> nenhum item encontrado na página {pagina}; fim da listagem.")
            break

        for item in itens_pagina:
            tag_a = item.select_one("h5.post-title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)

            resumo = None
            tag_resumo = item.select_one("p.from_the_blog_excerpt")
            if tag_resumo:
                resumo = tag_resumo.get_text(" ", strip=True)

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": url_item,
                "resumo": resumo,
            })

        print(f"  página {pagina}: {len(itens_pagina)} itens.")
        if len(itens_pagina) < ITENS_POR_PAGINA_ESPERADO:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_abrace()

print(f"\n{len(itens)} notícias listadas.\n")
for item in itens:
    print(f"- {item['titulo'][:80]}")

urls_unicas = {i["url"] for i in itens}
print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"\nExemplo de link: {itens[0]['url']}")
print(f"Exemplo de resumo: {itens[0]['resumo']}")

## Teste 2 — abrir uma notícia, extrair texto completo e descobrir a data

`.entry-content` (já no `SELETORES_CONTEUDO` compartilhado) pro texto.
Título de verdade em `h1.entry-title` (o *primeiro* `<h1>` da página é só
o cabeçalho genérico do template "Notícias" -- confirmando o achado
acima). Data em `time.entry-date[datetime]`, já em ISO 8601.

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    base = soup.select_one(".entry-content")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_titulo_verdadeiro(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.select_one("h1.entry-title")
    return h1.get_text(strip=True) if h1 else None


def extrair_data_publicacao(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    tag_time = soup.select_one("time.entry-date[datetime]")
    if tag_time and tag_time.get("datetime"):
        return tag_time["datetime"][:10]
    return None


def extrair_noticia_abrace(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)
    titulo_verdadeiro = extrair_titulo_verdadeiro(html)
    titulo_generico_h1 = BeautifulSoup(html, "lxml").find("h1")
    titulo_generico_h1 = titulo_generico_h1.get_text(strip=True) if titulo_generico_h1 else None

    return {
        "titulo": item["titulo"],
        "titulo_h1_generico_confere_bug": titulo_generico_h1,
        "titulo_h1_verdadeiro": titulo_verdadeiro,
        "url": item["url"],
        "published_at": extrair_data_publicacao(html),
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_abrace(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars, published_at={detalhe['published_at']}")
    print(f"    -> h1 generico (bug): {detalhe['titulo_h1_generico_confere_bug']!r} | h1.entry-title: {detalhe['titulo_h1_verdadeiro']!r}")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
sem_data = [d for d in detalhes if not d["published_at"]]
print(f"Com texto abaixo de 200 chars: {len(curtas)}")
print(f"Sem data: {len(sem_data)}")

In [0]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem paginada extrai título/resumo/link de
todos os itens, texto completo sai limpo via `.entry-content` (seletor já
compartilhado, sem precisar acrescentar nada). Confirmado o achado
esperado: **a data só aparece na página individual** (`time.entry-date`),
não na listagem -- e também um problema real que só apareceria em
produção se não fosse pego agora: o extrator genérico de título
(`extrair_titulo_h1`, primeiro `<h1>` da página) pegaria "Notícias"
(cabeçalho do template) em vez do título de verdade -- por isso a
integração usa `extrair_titulo: None` (mantém o título já certo, vindo da
listagem) em vez do extrator genérico.

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` -- sem Selenium, sem parsing que dependa de JS.
Precisa de uma `listar_abrace()` própria (paginação `/page/N`, seletores
do tema `post-item`/`post-title`) e um `extrair_data_abrace()` próprio
(le `time.entry-date[datetime]` na página individual) -- o texto reaproveita
`extrair_texto_generico()` sem seletor novo, e o título usa
`extrair_titulo: None` pelo motivo explicado acima.

Histórico médio (40 páginas / ~480 posts) -- `max_paginas` conservador
(recente, não backfill completo), mesmo critério de ANP/ABEGÁS/ABAR/Trata
Brasil.